# HW0: Berkeley Computing Check-In

**COMPSS 211A | Fall 2026 | Student copy**

Complete a small, reproducible Pandas workflow before graded course work begins.

**Date/deadline:** Sunday, September 6, 2026 at 11:59 p.m.

Run the provided code and replace each response placeholder with your own answer.

## Scenario

You are checking a small survey about how Berkeley graduate students travel to campus. Your job is to load the data, inspect it, produce a summary, save the result, and confirm that the saved file contains what you intended.

## What you will practice

- Locate course data without using a personal file path.
- Inspect a table before analyzing it.
- Summarize, export, and reopen a Pandas result.
- Check that an output is correct instead of assuming that successful code produced the intended result.

## Before you begin

If you are working locally, first run `lab/week00_installation_check.ipynb`. Continue when it displays **READY FOR WEEK 1**.

Make sure to open this notebook from within the course folder. Go to Explorer (file icon at the top left of the screen) -> Open Folder -> "compss-211a"

Then, run the following cell.

In [13]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )


def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None


LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)


def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path


print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

Python 3.13.15 | data=/Users/Angie/compss-211a/data


## 1. Record your GitHub username

Enter your GitHub username without the `@` symbol.

In [14]:
GITHUB_USERNAME = "ajweijiang"

if GITHUB_USERNAME.strip():
    print(f"GitHub username recorded: {GITHUB_USERNAME.strip()}")
else:
    print("Before submitting, enter your GitHub username above.")

GitHub username recorded: ajweijiang


## 2. Load and inspect the survey

Run the next cell before doing any analysis. It shows the table's dimensions, column types, missing values, and first three rows.

In [15]:
commute = pd.read_csv(course_data_path("hw1_after_dark_commute_survey.csv"))

print(f"Rows: {commute.shape[0]} | Columns: {commute.shape[1]}")
display(commute.dtypes.rename("dtype").to_frame())
display(commute.isna().sum().rename("missing_values").to_frame())
display(commute.head(3))

Rows: 96 | Columns: 9


,dtype
respondent_id,object
program,object
commute_mode,object
commute_minutes_one_way,float64
days_on_campus,int64
leaves_after_8pm_days,int64
monthly_transport_cost_usd,float64
wellbeing_score,float64
reliable_internet,object


,missing_values
respondent_id,0
program,0
commute_mode,0
commute_minutes_one_way,3
days_on_campus,0
leaves_after_8pm_days,0
monthly_transport_cost_usd,2
wellbeing_score,0
reliable_internet,0


,respondent_id,program,commute_mode,commute_minutes_one_way,days_on_campus,leaves_after_8pm_days,monthly_transport_cost_usd,wellbeing_score,reliable_internet
0,BMS-001,Information,Walk,18.0,5,2,0.00,6.8,Yes
1,BMS-002,Public Policy,Walk,13.0,4,2,0.00,6.2,Yes
2,BMS-003,Public Policy,Bike,16.0,5,1,92.27,7.5,Yes


### Your response

Which columns contain missing values, and how many values are missing from each?

> After looking at the results of the previously run cell it looks like we have missing values in columns 

"commute_minutes_one_way" with three missing values and

 "monthly_transport_cost_usd" with two missing values.

## 3. Summarize commute modes

The next cell counts respondents and calculates the median one-way commute time for each commute mode.

In [16]:
commute_summary = (
    commute.groupby("commute_mode", dropna=False)
    .agg(
        respondents=("respondent_id", "count"),
        median_one_way_minutes=("commute_minutes_one_way", "median"),
    )
    .reset_index()
    .sort_values("respondents", ascending=False)
)

display(commute_summary)

,commute_mode,respondents,median_one_way_minutes
1,BART + walk,23,54.0
5,Walk,21,18.0
0,AC Transit,19,42.0
2,Bike,13,24.5
3,Drive,11,32.0
4,Remote,9,0.0


### Your response

Which commute mode is most common in this survey? Report the mode and respondent count.

> The most common commute mode in this survey is "BART + walk" with 23 respondents.

In [17]:
display(commute_summary)

,commute_mode,respondents,median_one_way_minutes
1,BART + walk,23,54.0
5,Walk,21,18.0
0,AC Transit,19,42.0
2,Bike,13,24.5
3,Drive,11,32.0
4,Remote,9,0.0


## 4. Export and verify the result

Saving a file without an error does not prove that it contains the intended result. 

The next cell contains a common export mistake. It still runs and saves a file, but the reopened file is not quite what we intended. 

Run the cell, inspect the result, and then fix the export line so both checks print `True`. Do not change the checking code.

In [18]:
output_path = GENERATED_DIR / "hw0_commute_mode_summary.csv"
commute_summary.to_csv(output_path, index=False)

reopened_summary = pd.read_csv(output_path)
expected_columns = list(commute_summary.columns)
rows_match = len(reopened_summary) == len(commute_summary)
columns_match = list(reopened_summary.columns) == expected_columns

print(f"Saved: {output_path}")
print(f"Rows match: {rows_match}")
print(f"Columns match: {columns_match}")
if rows_match and columns_match:
    print("✓ Export verified")
else:
    print("✗ Check the exported file before continuing")

display(reopened_summary)

Saved: /Users/Angie/compss-211a/generated/hw0_commute_mode_summary.csv
Rows match: True
Columns match: True
✓ Export verified


,commute_mode,respondents,median_one_way_minutes
0,BART + walk,23,54.0
1,Walk,21,18.0
2,AC Transit,19,42.0
3,Bike,13,24.5
4,Drive,11,32.0
5,Remote,9,0.0


## 5. Troubleshooting response

What extra column appeared in the reopened file? What change did you make to the export line so the saved file contained only the intended columns?

> In the original export line, python made a new column from the dataframe's index, using the index for values. When we tried to check if they were the same data frames, this extra column caused the check to fail. If we add the argument "index=FALSE", the export line will export without creating a new column for the index. When we read the csv back in, the check will return True for columns.